In [0]:
catalog = "workspace"
schema = "certification_pipeline_lab"

spark.sql(f"""
CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}
""")

spark.sql(f"""
CREATE VOLUME IF NOT EXISTS {catalog}.{schema}.raw_files
""")

In [0]:
volume_root = (
    f"/Volumes/{catalog}/{schema}/raw_files"
)

orders_path = f"{volume_root}/orders"
status_path = f"{volume_root}/status"
customers_cdc_path = f"{volume_root}/customers_cdc"

print(volume_root)

In [0]:
for path in [
    orders_path,
    status_path,
    customers_cdc_path
]:
    dbutils.fs.rm(path, recurse=True)

In [0]:
from pyspark.sql import Row
from pyspark.sql import functions as F

orders = [
    Row(
        order_id=1001,
        order_timestamp="2026-09-01T10:15:00",
        customer_id=1,
        country_code="FR",
        amount=120.50,
        notifications="Y"
    ),
    Row(
        order_id=1002,
        order_timestamp="2026-09-01T11:30:00",
        customer_id=2,
        country_code="DE",
        amount=89.90,
        notifications="N"
    ),
    Row(
        order_id=1003,
        order_timestamp="2026-09-02T09:00:00",
        customer_id=1,
        country_code="FR",
        amount=250.00,
        notifications="Y"
    ),
    Row(
        order_id=1004,
        order_timestamp="2026-09-02T14:20:00",
        customer_id=3,
        country_code="ES",
        amount=45.00,
        notifications="INVALID"
    ),
    Row(
        order_id=1005,
        order_timestamp="2026-09-03T16:45:00",
        customer_id=None,
        country_code="IT",
        amount=75.25,
        notifications="Y"
    )
]

orders_df = spark.createDataFrame(orders)

(
    orders_df
    .coalesce(1)
    .write
    .mode("overwrite")
    .json(f"{orders_path}/batch_001")
)

display(
    spark.read.json(f"{orders_path}/batch_001")
)

In [0]:
new_order = [
    (
        2001,
        "2026-09-04T10:00:00",
        2,
        "DE",
        149.99,
        "Y"
    )
]

new_order_df = spark.createDataFrame(
    new_order,
    [
        "order_id",
        "order_timestamp",
        "customer_id",
        "country_code",
        "amount",
        "notifications"
    ]
)

(
    new_order_df
    .coalesce(1)
    .write
    .mode("overwrite")
    .json(
        f"{orders_path}/batch_002"
    )
)

In [0]:
'''statuses = [
    Row(
        order_id=1001,
        status="DELIVERED",
        status_timestamp="2026-09-01T14:00:00"
    ),
    Row(
        order_id=1002,
        status="CANCELLED",
        status_timestamp="2026-09-01T12:15:00"
    ),
    Row(
        order_id=1003,
        status="SHIPPED",
        status_timestamp="2026-09-02T13:30:00"
    ),
    Row(
        order_id=1004,
        status="DELIVERED",
        status_timestamp="2026-09-03T09:20:00"
    )
]'''

statuses = [
    Row(
        order_id=1001,
        status="SHIPPED",
        status_timestamp="2026-08-01T14:00:00"
    ),
    Row(
        order_id=1002,
        status="CREATED",
        status_timestamp="2026-08-20T12:15:00"
    ),
    Row(
        order_id=1003,
        status="DELIVERED",
        status_timestamp="2026-10-02T13:30:00"
    ),
    Row(
        order_id=1004,
        status="CREATED",
        status_timestamp="2026-08-15T09:20:00"
    )
]

status_df = spark.createDataFrame(statuses)

(
    status_df
    .coalesce(1)
    .write
    #.mode("overwrite")
    .mode("append")
    .json(f"{status_path}/batch_001")
)

display(
    spark.read.json(f"{status_path}/batch_001").orderBy("order_id","status_timestamp")
)

In [0]:
'''new_status = [
    (
        2001,
        "SHIPPED",
        "2026-09-04T12:00:00"
    )
]'''

new_status = [
    (
        2001,
        "DELIVERED",
        "2026-09-05T12:30:00"
    )
]

new_status_df = spark.createDataFrame(
    new_status,
    [
        "order_id",
        "status",
        "status_timestamp"
    ]
)

(
    new_status_df
    .coalesce(1)
    .write
    .mode("overwrite")
    #.mode("append")
    .json(
        f"{status_path}/batch_003"
    )
)

In [0]:
customer_events = [
    Row(
        customer_id=1,
        name="Alice",
        country="FR",
        email="alice@example.com",
        operation="INSERT",
        sequence=1
    ),
    Row(
        customer_id=2,
        name="Bob",
        country="DE",
        email="bob@example.com",
        operation="INSERT",
        sequence=1
    ),
    Row(
        customer_id=3,
        name="Carlos",
        country="ES",
        email="carlos@example.com",
        operation="INSERT",
        sequence=1
    )
]

customer_events_df = spark.createDataFrame(customer_events)

(
    customer_events_df
    .coalesce(1)
    .write
    .mode("overwrite")
    .json(f"{customers_cdc_path}/batch_001")
)

display(
    spark.read
    #.option("recursiveFileLookup", "true")
    .json(f"{customers_cdc_path}/batch_001")
)

In [0]:
customer_changes_batch_002 = [
    (
        1,
        "Alice",
        "BE",
        "alice@example.com",
        "UPDATE",
        3
    ),

    (
        2,
        "Bob",
        "DE",
        "bob.new@example.com",
        "UPDATE",
        2
    ),

    (
        3,
        "Carlos",
        "ES",
        "carlos@example.com",
        "DELETE",
        2
    ),

    (
        4,
        "Diana",
        "IT",
        "diana@example.com",
        "INSERT",
        1
    )
]

customer_changes_df = spark.createDataFrame(
    customer_changes_batch_002,
    [
        "customer_id",
        "name",
        "country",
        "email",
        "operation",
        "sequence"
    ]
)

(
    customer_changes_df
    .coalesce(1)
    .write
    .mode("overwrite")
    .json(
        f"{customers_cdc_path}/batch_002"
    )
)

In [0]:
late_customer_change = [
    (
        1,
        "Alice",
        "DE",
        "alice@example.com",
        "UPDATE",
        2
    )
]

late_customer_change_df = spark.createDataFrame(
    late_customer_change,
    [
        "customer_id",
        "name",
        "country",
        "email",
        "operation",
        "sequence"
    ]
)

(
    late_customer_change_df
    .coalesce(1)
    .write
    .mode("overwrite")
    .json(
        f"{customers_cdc_path}/batch_003"
    )
)

In [0]:
countries = [
    ("FR", "France", "Paris"),
    ("DE", "Germany", "Berlin"),
    ("ES", "Spain", "Madrid"),
    ("IT", "Italy", "Rome"),
    ("BE", "Belgium", "Brussels")
]

countries_df = spark.createDataFrame(
    countries,
    ["country_code", "country_name", "capital"]
)

(
    countries_df
    .write
    .mode("overwrite")
    .saveAsTable(
        f"{catalog}.{schema}.ref_countries"
    )
)

display(
    spark.table(
        f"{catalog}.{schema}.ref_countries"
    )
)

In [0]:
files = []

for path in [orders_path, status_path, customers_cdc_path]:
    files += dbutils.fs.ls(path)

display(files)

display(
    spark.table(
        "workspace.certification_pipeline_lab.ref_countries"
    )
)